In [1]:
import torch
import os
from dotenv import load_dotenv

%env LDFLAGS=-L/opt/homebrew/opt/jpeg/lib
%env CPPFLAGS=-I/opt/homebrew/opt/jpeg/include
%env LDFLAGS="-L/opt/homebrew/opt/zlib/lib -L/opt/homebrew/opt/jpeg/lib"
%env CPPFLAGS="-I/opt/homebrew/opt/zlib/include -I/opt/homebrew/opt/jpeg/include"
%env PKG_CONFIG_PATH="/opt/homebrew/opt/zlib/lib/pkgconfig:/opt/homebrew/opt/jpeg/lib/pkgconfig"

!pip install pillow llama-index llama-cpp-python python-dotenv pandas jupyter pymupdf llama-index-embeddings-huggingface llama-index-llms-llama-cpp
!pip install -q -U google-genai


env: LDFLAGS=-L/opt/homebrew/opt/jpeg/lib
env: CPPFLAGS=-I/opt/homebrew/opt/jpeg/include
env: LDFLAGS="-L/opt/homebrew/opt/zlib/lib -L/opt/homebrew/opt/jpeg/lib"
env: CPPFLAGS="-I/opt/homebrew/opt/zlib/include -I/opt/homebrew/opt/jpeg/include"
env: PKG_CONFIG_PATH="/opt/homebrew/opt/zlib/lib/pkgconfig:/opt/homebrew/opt/jpeg/lib/pkgconfig"


In [2]:
import os
from google import genai
from google.genai import types

GEMINI_API_KEY = ""

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing. Please set it in your environment or a .env file.")

# 1. Initialize client
client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client initialized successfully!")


Gemini client initialized successfully!


In [3]:
from llama_cpp import Llama
import os

model_path = "content/mistral.gguf"
os.makedirs("content", exist_ok=True)
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")

llm = Llama(
    model_path=model_path,
    n_gpu_layers=1,
    n_ctx=2048,
    verbose=False
)
print("Mistral model loaded successfully!")


Model file exists. Size: 4166.07 MB
Mistral model loaded successfully!


In [6]:
import time

!pip install pymupdf
!pip install llama-index-llms-llama-cpp
!pip install llama-index-embeddings-huggingface



In [9]:
import fitz  # PyMuPDF

# Load the sample contract PDF
pdf_path = "sample-sdf-document.pdf"
doc = fitz.open(pdf_path)

# Extract text from all pages
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")

Extracted 617 words from the PDF.


In [11]:
from llama_index.core import VectorStoreIndex, Document, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure the LLM
llm = LlamaCPP(
    model_path="content/mistral.gguf",
    temperature=0.7,
    max_new_tokens=512,
    context_window=2048,
    model_kwargs={"n_gpu_layers": 1}
)

# Configure open-source embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"  # Lightweight but effective embedding model
)

# Set as the default LLM and embedding model
Settings.llm = llm
Settings.embed_model = embed_model

# Create documents from your text
documents = [Document(text=text)]  # 'text' should be your document content

# Build index
index = VectorStoreIndex.from_documents(documents)

# Configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2,  # Retrieve 2 most similar chunks
)

# Configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)


llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.h

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
query = "What are the late payment penalties in this contract?"

# Assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)


# Test query
response = query_engine.query(query)
print(response)

llama_perf_context_print:        load time =   11127.01 ms
llama_perf_context_print: prompt eval time =   11126.53 ms /  1286 tokens (    8.65 ms per token,   115.58 tokens per second)
llama_perf_context_print:        eval time =     613.09 ms /    15 runs   (   40.87 ms per token,    24.47 tokens per second)
llama_perf_context_print:       total time =   11741.31 ms /  1301 tokens
llama_perf_context_print:    graphs reused =         14


 Unfortunately, the provided information does not mention anything about late payment penalties.
